# Gapped motifs

Most motifs are short, contiguous stretches of DNA, and most of the package
is written with those in mind. Some binding sites are made of two conserved
halves separated by a spacer whose length varies from one site to the next,
however, such as the direct and inverted repeats bound by many dimeric
transcription factors. This guide shows how to describe such sites as gapped
motifs, scan sequences with them, compare and merge them, and find them
*de novo*.

As usual, everything comes from the top-level package:

In [ ]:
import pathlib
import tempfile

import polars as pl
import universalmotif as um

## Adding gaps to a motif

Let us consider a motif made of two copies of the half-site AGGTCA. Proteins
that bind such direct repeats often tolerate a spacer of a few bases between
the two halves, and its length can vary from one site to the next. An
ordinary motif can only describe one spacing at a time. To describe all of
them at once, we can add a gap with [`add_gap()`][universalmotif.add_gap].
The gap comes after the first `gaploc` positions, and can span anywhere from
`mingap` to `maxgap` positions. Here the gap sits between the two half-sites
and spans one to five positions:

In [ ]:
dr = um.create_motif("AGGTCAAGGTCA", name="DR", nsites=20, pseudocount=1)
dr_gapped = um.add_gap(dr, gaploc=6, mingap=1, maxgap=5)
print(dr_gapped)

The matrix itself is unchanged, since the gap has no columns of its own.
The printed consensus marks where the gap goes with two dots, whatever its
size. Tables from
[`gapped_motif_finder()`][universalmotif.gapped_motif_finder] write it in
their `consensus` column as `[min-max]` instead, so this motif would read
`AGGTCA[1-5]AGGTCA` there. The gap's location and range of sizes are listed
above the matrix, and are available as attributes too.

In [ ]:
print(dr_gapped.isgapped, dr_gapped.gaploc, dr_gapped.mingap, dr_gapped.maxgap)

A motif can have more than one gap. In that case `gaploc`, `mingap` and
`maxgap` take one value per gap, and shorter lists are recycled. A gap may
also be allowed a size of zero, which means the motif can be matched with or
without it. For example, here is a version of the same motif with an
additional optional gap after the third position:

In [ ]:
dr_two_gaps = um.add_gap(dr, gaploc=[3, 6], mingap=[0, 1], maxgap=[1, 5])
print(dr_two_gaps.gaploc, dr_two_gaps.mingap, dr_two_gaps.maxgap)

This motif can be matched in 2 × 5 = 10 different ways, one for each
combination of gap sizes. This will become important later when we look at
P-values.

## Scanning gapped motifs

Now let us make a few random sequences and plant a direct repeat in each.
The first three get a spacer of one, two and four bases. The fourth gets a
repeat with a three base spacer, planted as its reverse complement so that
it can only be found on the reverse strand:

In [ ]:
def plant(seq, site, at=90):
    return seq[:at] + site + seq[at + len(site):]

sites = {
    "seq1": "AGGTCA" + "A" + "AGGTCA",
    "seq2": "AGGTCA" + "GA" + "AGGTCA",
    "seq3": "AGGTCA" + "CATG" + "AGGTCA",
    "seq4": "TGACCT" + "GAC" + "TGACCT",
}
random_seqs = um.create_sequences(seqnum=4, seqlen=200, seed=4)
seqs = {name: plant(s, site) for (name, site), s in zip(sites.items(), random_seqs)}

The original motif, which has no gap, expects the two half-sites to sit
right next to each other, so it finds none of them.

In [ ]:
um.scan_sequences(dr, seqs, pvalue=1e-5).height

The gapped motif, on the other hand, finds all four. When given a gapped
motif, [`scan_sequences()`][universalmotif.scan_sequences] tries every
combination of gap sizes at every position:

In [ ]:
cols = ["sequence", "start", "end", "strand", "score", "match", "pvalue"]
hits = um.scan_sequences(dr_gapped, seqs, pvalue=1e-5)
hits.select(cols)

The letters skipped over by the gap show as dots, so counting them gives the
spacer of each hit. This is a quick way to see which spacings a set of sites
favours:

In [ ]:
hits.select("sequence", "match").with_columns(gap=pl.col("match").str.count_matches(r"\."))

Gapped motifs are scanned on both strands like any other DNA motif. The site
in seq4 is therefore found on the `-` strand, with its match shown in the
orientation of the motif. Note also that `start` and `end` include the gap,
which is why the hits are of different lengths.

The letters inside the gap are not scored at all, so every hit here has the
same score and the same P-value. That P-value is not quite the one
[`motif_pvalue()`][universalmotif.motif_pvalue] gives for the score under
the ungapped motif, however. It is five times larger, for a reason we will
come to shortly:

In [ ]:
um.motif_pvalue(dr, score=hits["score"][0])

To scan a gapped motif as if it had no gaps, set `use_gaps=False`. Here,
that gives the same result as scanning the original motif.

In [ ]:
um.scan_sequences(dr_gapped, seqs, pvalue=1e-5, use_gaps=False).height

To turn the gaps off for good, use [`ungap()`][universalmotif.ungap]. The
gap locations and sizes are kept in the motif's attributes, so they can be
passed back to [`add_gap()`][universalmotif.add_gap] later. Setting
`delete=True` deletes them instead:

In [ ]:
off = um.ungap(dr_gapped)
print(off.isgapped, off.gaploc, off.mingap, off.maxgap)
print(um.add_gap(off, off.gaploc, off.mingap, off.maxgap) == dr_gapped)
gone = um.ungap(dr_gapped, delete=True)
print(gone.isgapped, gone.gaploc)

## Correcting P-values for gaps

There is a catch to scanning gapped motifs, however. Each combination of
gap sizes is effectively a different motif, and each gets its own chance to
produce a hit at every position. A motif with five possible gap sizes is
thus tested five times as often as one without a gap. By chance alone it
can give up to five times as many hits at the same P-value cutoff. The
P-values themselves are not wrong, as each is the P-value of a single
match, but they do not account for the number of matches tried.

The `correct_gaps` argument applies a Bonferroni correction for this. Each
P-value is multiplied by the number of gap size combinations and capped at
1, and hits must reach the cutoff after correction. It is on by default,
which is why the P-values of the planted sites above were five times those
of their scores, though still far below the cutoff. With
`correct_gaps=False`, each hit gets the P-value of its score instead:

In [ ]:
um.scan_sequences(dr_gapped, seqs, pvalue=1e-5, correct_gaps=False).select(cols)

For strong sites like these the correction changes little. It matters more
for chance hits in long sequences, particularly when gaps are allowed to
vary a lot. To see this, let us scan the 100 background sequences from the
example data, which contain no planted direct repeats. We scan them with
the ungapped motif and with two gapped versions of it, one with the gap
varying from one to five positions and another from zero to twenty. Every
hit in these sequences is a false positive:

In [ ]:
controls = um.example_sequences("controls")
rows = []
for label, motif in [
    ("no gap", dr),
    ("gap of 1-5", um.add_gap(dr, gaploc=6, mingap=1, maxgap=5)),
    ("gap of 0-20", um.add_gap(dr, gaploc=6, mingap=0, maxgap=20)),
]:
    rows.append({
        "motif": label,
        "hits": um.scan_sequences(motif, controls, pvalue=1e-4, correct_gaps=False).height,
        "hits_corrected": um.scan_sequences(motif, controls, pvalue=1e-4).height,
    })
pl.DataFrame(rows)

Without the correction, the motif with a gap of one to five positions finds
three times as many chance hits as the ungapped motif. The motif whose gap
spans 21 possible sizes finds twenty times as many. With the correction,
almost all of them are gone. The motif without a gap is unaffected by
`correct_gaps`.

The corrected P-values are the fairer ones when looking for the best
candidate sites. They are also fairer when comparing hit counts between
gapped motifs with different gap ranges, or with ungapped motifs, which is
why they are the default. If you only want to know which of a handful of
sequences contain a gapped site, the uncorrected P-values are fine. Just
keep in mind that a wide gap range makes chance hits more likely. Either
way it is best to keep gap ranges no wider than needed.

## Comparing and merging gapped motifs

[`compare_motifs()`][universalmotif.compare_motifs] compares a gapped motif
as each of the motifs it stands for, one for every combination of its gap
sizes, with each gap filled by columns of the background. It keeps the best
of these comparisons, multiplies its P-value by the number tried, and gives
the gap sizes it used in `subject_gaps` and `target_gaps`. Let us compare
the direct repeat, with its spacer of one to five letters, against two
ungapped motifs with a fixed spacer of four: another direct repeat, and an
inverted repeat.

In [ ]:
dr4 = um.create_motif("AGGTCANNNNAGGTCA", name="DR4", nsites=20, pseudocount=1)
ir4 = um.create_motif("AGGTCANNNNTGACCT", name="IR4", nsites=20, pseudocount=1)
compared = um.compare_motifs([dr_gapped, dr4, ir4], compare_to=0)
compared.filter(pl.col("target") != "DR").select("target", "overlap", "score", "pvalue", "subject_gaps")

With a gap of four letters, the direct repeat lines up with DR4 over both of
its halves, while IR4 matches only one of them. With `use_gaps=False` the
gap is ignored and the two halves run together. Neither fixed-spacer motif
then fits beyond a single half, and the two can no longer be told apart:

In [ ]:
compared = um.compare_motifs([dr_gapped, dr4, ir4], compare_to=0, use_gaps=False)
compared.filter(pl.col("target") != "DR").select("target", "overlap", "score", "pvalue")

[`merge_motifs()`][universalmotif.merge_motifs] aligns gapped motifs in the
same way. When the motifs merged have the same number of gaps, they are
merged part by part, and the merged motif keeps a gap spanning the sizes of
all of them. Here a direct repeat of slightly different half-sites with a
fixed spacer of seven merges with the one above into a motif whose gap spans
one to seven letters:

In [ ]:
dr7 = um.add_gap(
    um.create_motif(["AGGTCAAGGTCA", "AGTTCAAGGTCA", "GGGTCAAGTTCA"], name="DR7", pseudocount=1),
    gaploc=6, mingap=7, maxgap=7,
)
merged = um.merge_motifs([dr_gapped, dr7])
print(merged.consensus, merged.gaploc, merged.mingap, merged.maxgap)

Gapped motifs with different numbers of gaps can be merged too, but each
then goes in at its best-fitting gap sizes and the merged motif has no gaps,
with a warning saying so.

## Finding gapped motifs

[`motif_finder()`][universalmotif.motif_finder] finds a gapped site only as
its half-site, since the varying spacer spreads the full site over many
different words.
[`gapped_motif_finder()`][universalmotif.gapped_motif_finder] instead counts
pairs of short words at every spacing up to `max_gap`. It builds gapped
motifs from the most enriched pairs and keeps those whose hits favour some
spacings over others. The finder is experimental, so its options and results
may change in later releases. Let us plant an inverted repeat of TGTCGG with
a spacer of 6 to 9 letters in 200 AT-rich sequences, keeping another 200 as
controls:

In [ ]:
at_rich = {"A": 0.32, "C": 0.18, "G": 0.18, "T": 0.32}
ir = um.add_gap(um.create_motif("TGTCGGCCGACA", name="IR"), gaploc=6, mingap=6, maxgap=9)
seqs = um.create_sequences(seqnum=400, seqlen=300, freqs=at_rich, seed=20)
targets = um.implant_motifs(ir, seqs[:200], seed=21)
controls = seqs[200:]
found = um.gapped_motif_finder(targets, controls, seed=1, nthreads=2)
found.select("consensus", "seqs_pos", "seqs_neg", "log10_pvalue", "log10_spacing_pvalue")

The motif comes back with its spacer of 6 to 9 letters, and nothing else
was found. The search runs in a few steps:

1. A share of the sequences and of the controls, `holdout`, is set aside to
   test the motifs found in the rest.
2. Pairs of k-mers of 4, 5 and 6 letters are counted at every spacing from
   `min_gap` to `max_gap`, in the sequences and in the controls. The pairs
   most enriched over the controls seed motifs.
3. Each seed is built into a gapped motif from its sites and refined by
   rescanning. It keeps the spacer sizes its hits favour, and each of its
   parts grows into neighbouring informative columns.
4. Each motif is tested in the sequences held out. Its hits must favour its
   spacings over the two parts falling anywhere, and sequences with hits must
   be enriched over the controls.
5. The best motif to pass is kept, its hits are masked, and the search
   starts again, until `nmotifs` motifs are found or none passes.

As the tests use sequences the motifs were not fitted to, their P-values are
not flattered by the search itself, however many pairs it tries. Two columns
hold them:

- `log10_spacing_pvalue` tests whether the hits favour these spacings over
  the two parts falling anywhere.
- `log10_pvalue` tests whether sequences with hits are enriched over the
  controls.

The table also gives each motif's parts and gap (`left`, `right`, `min_gap`
and `max_gap`) and the pair of k-mers it grew from (`seed_pair`). The motif
itself can be drawn and scanned like any other gapped motif. Counting the
gap sizes of its hits gives back the planted spacers:

In [ ]:
motif = found["motif"][0]
fig = um.view_motifs(motif)

In [ ]:
(
    um.scan_sequences(motif, targets, no_overlaps=True)
    .with_columns(gap=pl.col("match").str.count_matches(r"\."))
    .group_by("gap").len().sort("gap")
)

On the same sequences, `motif_finder()` sees only the half-site:

In [ ]:
um.motif_finder(targets, controls, min_width=6, max_width=12, seed=1, nthreads=2).select(
    "consensus", "width", "seqs_pos", "seqs_neg", "pvalue"
)

### Options

The defaults suit a few hundred to a few thousand peaks of a few hundred
letters. These are the options most worth knowing:

| Option | Default | What it does |
|---|---|---|
| `max_gap` | 31 | The longest spacer searched, at most 31, so that the motifs found can still be compared and merged with their gaps |
| `nmotifs` | 5 | The most motifs to find |
| `k` | 4, 5, 6 | The k-mer sizes that seed motifs, each from 2 to 6 |
| `min_fraction` | 0.02 | The share of the sequences a pair of k-mers must be in to be tested; lower values find rarer motifs, at the cost of more tests to correct for |
| `holdout` | 0.3 | The share of the sequences and controls held out to test the motifs; with 0 they are tested where they were found, and the P-values are somewhat optimistic |
| `mismatch` | `"auto"` | Whether pairs of 6-mers also count with one mismatch, which helps with halves too variable to be counted exactly but takes several times longer; `"auto"` does so only when the exact search finds nothing |
| `mask` | `True` | Masks low-complexity regions with DUST before the search, as their repeated units otherwise look like parts with a favoured spacing |
| `stop_pvalue` | 0.001 | The spacing P-value a motif must reach to be kept |

With `mismatch="auto"`, the second search keeps motifs at half of
`stop_pvalue`, so that searching twice in sequences with nothing to find
reports no more false motifs than searching once.

### Backgrounds

With peaks from a genome, a background drawn from the genome with
[`match_bkg()`][universalmotif.match_bkg] matters even more here than in
enrichment tests. Shuffled sequences, which the finder uses when no controls
are given, lose the loose repeats that genomes are full of, and the repeated
units of those look like halves with a favoured spacing. The
[Choosing a background](background-models.ipynb) guide shows how to draw
one.

### Saving gapped motifs

Most motif file formats have no place for gaps. The MEME, JASPAR and TRANSFAC
writers therefore give a gapped motif its gaps as its alternative name,
written into its consensus as in the finder's table, and the readers turn
that name back into gaps. Other tools see an ordinary motif, with its parts
joined and an alternative name that shows where its gap goes:

In [ ]:
path = pathlib.Path(tempfile.mkdtemp()) / "gapped.meme"
um.write_meme(found["motif"].to_list(), path)
print([line for line in path.read_text().splitlines() if line.startswith("MOTIF")])

back = um.read_meme(path)[0]
print(back.gaploc, back.mingap, back.maxgap)

A motif's own alternative name gives way to its gaps, with a warning.
HOMER, CIS-BP, UniPROBE and plain matrix files lose the gaps altogether.
[`write_universalmotif()`][universalmotif.write_universalmotif] keeps them
along with every other field of the motif, in a format that
[`read_universalmotif()`][universalmotif.read_universalmotif] reads back
whole.

On the command line, `umotif find-gapped` runs the finder and writes the
table alongside the motifs, as the [Command line](../command-line.md) page
shows.

## Where next

The [Scanning large genomes](large-genomes.ipynb) guide shows how to scan
genome-sized inputs efficiently. The
[API reference](../reference/index.md) describes every function used here
in more detail.